# Web Scraping con Selenium



##### Extrae de las películas en cartelera datos. De ellas vamos a extraer la siguiente información:
- ##### Fecha de estreno
- ##### URL
- ##### Datos principales, como hemos hecho al principio
- ##### Nota media
- ##### Cantidad de votos
- ##### Críticas profesionales buenas, regulares y malas
- ##### Cinco primeras críticas

Importación de las librerías

In [1]:
# Para la manipulación de datos
import pandas as pd

# Servicio y driver de Chrome de Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Las opciones que vamos a tener para buscar elementos
from selenium.webdriver.common.by import By

# Para cuando queramos mandar pulsaciones de teclado
from selenium.webdriver.common.keys import Keys

# Hacemos que espere
import time

# Importaciones para esperas explícitas (mejor práctica que time.sleep)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Importamos undetected-chromedriver para evitar el captcha
import undetected_chromedriver as uc

# Importamos excepciones comunes de Selenium para manejo de errores
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

In [2]:
from bs4 import BeautifulSoup as bs

service = Service(ChromeDriverManager().install())

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
print(driver.capabilities["browserVersion"])
driver.quit()

146.0.7680.154


In [4]:
from pprint import pprint

Creamos el driver para controlar el navegador

In [5]:
import undetected_chromedriver as uc

driver = uc.Chrome(
    browser_executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    service=service,
    use_subprocess=False,
    headless=False,
)

RECUERDA:

##### Beginner Selenium Cheatsheet:
Sacar un elemento:
- element = driver.find_element(by, value)

Sacar varios elementos:
- element = driver.find_elements(by, value)

Sacar atributos:
- attribute = element.--el atributo--
- attribute = element.get_attribute(--el atributo--)

Hacer click:
- element.click()

Teclear:
- element.send_keys()

Gestión de pestañas:
- driver.switch_to.window(driver.window_handles[-1])
- driver.get(url)
- driver.close()

Accedemos a la página principal

In [6]:

url = "https://www.filmaffinity.com"
driver.get(url)

Aceptamos el pop-up de ser necesario

In [7]:
elements_by_tag = driver.find_elements(By.TAG_NAME, 'button')

for i in elements_by_tag:
    print(i.text)
    if i.text == 'ACEPTO':
        i.click()


MÁS OPCIONES
ACEPTO




Hacemos una función que devuelva en un diccionario todos los datos de las películas, salvo la fecha de estreno y la url

Parámetros: url y fecha de estreno
Salida: Diccionario con todos los datos

In [8]:
# Consigo el enlace de las top peliculas

search = driver.find_element(By.CLASS_NAME, 'news-wrapper')
search = search.find_element(By.ID, 'awardsmenu')
search = search.find_element(By.TAG_NAME, 'a')
link = search.get_attribute('href')

print(link)

https://www.filmaffinity.com/es/topgen.php


In [9]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)

try:
    duration = wait.until(
        EC.presence_of_element_located((By.XPATH, '//dd[@itemprop="duration"]'))
    ).text
except:
    duration = "N/A"

In [10]:
driver.get(link)

top_movies = driver.find_elements(By.ID, 'top-movies')

for movie in top_movies:
    position = movie.find_element(By.CLASS_NAME, 'position')
    title = movie.find_element(By.CLASS_NAME, 'mc-title').find_element(By.TAG_NAME, 'a').get_attribute('title')
    href = movie.find_element(By.CLASS_NAME, 'mc-title').find_element(By.TAG_NAME, 'a').get_attribute('href')
    print(href)
    driver.get(href)
    title2 = driver.find_element(By.CLASS_NAME, 'movie-info').find_element(By.TAG_NAME, 'dd').text
    print(title2)
    year = driver.find_element(By.XPATH, '//dd[@itemprop="datePublished"]').text
    print(year)
    elements = driver.find_elements(By.XPATH, '//dd[@itemprop="duration"]')
    if elements:
        duration = elements[0].text
    else:
        duration = "N/A"

    print(duration)
    country = driver.find_element(By.XPATH, '//span[@id="country-img"]').find_element(By.XPATH, '//img[@class="nflag"]').get_attribute('alt')

    a = driver.find_element(By.XPATH, '//a[@itemprop="url" and @class="link"]')
    span = a.find_element(By.XPATH, './/span[@itemprop="name"]')
    director = span.text

    # if elements:
    #     director = elements[0].text
    # else:
    #     director = "N/A"
    # director = driver.find_element(By.CLASS_NAME, 'credits').find_element(By.XPATH, '//a[class="link"]').get_attribute('title')
    print(country)
    print(director)





https://www.filmaffinity.com/es/film809297.html
The Godfather
1972
175 min.
Estados Unidos
Francis Ford Coppola


In [ ]:
def get_movie(url):
    movie_details = []
    driver.get(url)
    time.sleep(0.1)
    
    title = driver.find_element(By.CLASS_NAME, 'movie-info').find_element(By.TAG_NAME, 'dd').text
    
    year = driver.find_element(By.XPATH, '//dd[@itemprop="datePublished"]').text
    
    elements = driver.find_elements(By.XPATH, '//dd[@itemprop="duration"]')
    if elements:
        duration = elements[0].text
    else:
        duration = "N/A"

    
    country = driver.find_element(By.XPATH, '//span[@id="country-img"]').find_element(By.XPATH, '//img[@class="nflag"]').get_attribute('alt')

    a = driver.find_element(By.XPATH, '//a[@itemprop="url" and @class="link"]')
    span = a.find_element(By.XPATH, './/span[@itemprop="name"]')
    director = span.text
    rating = "N/A"
    try:
        # rating = driver.find_element(By.XPATH, '//div[contains(@id, "movie-rate-avg") and @itemprop="ratingValue"]').text.strip()
        # rating = driver.find_element(By.CSS_SELECTOR, '#movie-rat-avg')
        time.sleep(1)
        rating = driver.find_element(By.XPATH, '//div[@itemprop="ratingValue"]').text

    except Exception as e:
        print(f"Error en rating: {e}")


    # print(title)
    # print(year)
    # print(duration)
    # print(country)
    # print(director)
    movie_det = {'atitle': title, 'year': year, 'country': country, 'director': director, 'duration': duration, 'rating': rating}
    movie_details.append(movie_det)

    pprint(movie_details)
    return movie_details


    
    

In [12]:
# get_movie('https://www.filmaffinity.com/es/film809297.html')

Probamos la función que hemos hecho. Aquí tienes un enlace de prueba: https://www.filmaffinity.com/es/film599984.html

In [13]:
prueba = get_movie("https://www.filmaffinity.com/es/film618375.html")
prueba

[{'country': 'Estados Unidos',
  'director': 'Joseph Kosinski',
  'duration': '126 min.',
  'rating': '5,8',
  'title': 'Oblivion',
  'year': '2013'}]


[{'title': 'Oblivion',
  'year': '2013',
  'country': 'Estados Unidos',
  'director': 'Joseph Kosinski',
  'duration': '126 min.',
  'rating': '5,8'}]

In [14]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import undetected_chromedriver as uc
import time


options = uc.ChromeOptions()
options.headless = False
driver = uc.Chrome(options=options)

url = "https://www.filmaffinity.com"
driver.get(url)

elements_by_tag = driver.find_elements(By.TAG_NAME, 'button')

for i in elements_by_tag:
    print(i.text)
    if i.text == 'ACEPTO':
        i.click()

url = "https://www.filmaffinity.com/es/topgen.php"
href_list = []
title_list = []
driver.get(url)
time.sleep(3)  


for i in range(1, 11):  
    try:
        card = driver.find_element(By.CSS_SELECTOR, f"ul#top-movies li:nth-child({i}) div.movie-card")
        title_el = card.find_element(By.CSS_SELECTOR, "div.mc-title a")
        
        title = title_el.text
        href = title_el.get_attribute("href")
        print(f"{i}. {title} | {href}")
        title_list.append(title)
        href_list.append(href)
    
        
        time.sleep(0.2)  
        
    except (NoSuchElementException, StaleElementReferenceException):
        print(f"Error en posición {i}")
        continue
all_movies = []
for href in href_list:
    get_movie(href)
    all_movies.append(get_movie(href))





MÁS OPCIONES
ACEPTO


1. El padrino | https://www.filmaffinity.com/es/film809297.html
2. El padrino | https://www.filmaffinity.com/es/film809297.html
3. El padrino. Parte II | https://www.filmaffinity.com/es/film730528.html
4. The Wire (Bajo escucha) | https://www.filmaffinity.com/es/film399474.html
5. Breaking Bad | https://www.filmaffinity.com/es/film489970.html
Error en posición 6
7. Cosmos | https://www.filmaffinity.com/es/film601451.html
8. Planeta azul II | https://www.filmaffinity.com/es/film518489.html
9. Doce hombres sin piedad | https://www.filmaffinity.com/es/film695552.html
10. Cosmos: A Space-Time Odyssey | https://www.filmaffinity.com/es/film661074.html
11. La lista de Schindler | https://www.filmaffinity.com/es/film656153.html
Error en posición 12
13. Testigo de cargo | https://www.filmaffinity.com/es/film667376.html
14. Cadena perpetua | https://www.filmaffinity.com/es/film161026.html
15. Apocalipsis: La Segunda Guerra Mundial | https://www.filmaffinity.com/es/film22147

In [16]:
pprint(all_movies)

[[{'country': 'Estados Unidos',
   'director': 'Francis Ford Coppola',
   'duration': '175 min.',
   'rating': '9,0',
   'title': 'The Godfather',
   'year': '1972'}],
 [{'country': 'Estados Unidos',
   'director': 'Francis Ford Coppola',
   'duration': '175 min.',
   'rating': '9,0',
   'title': 'The Godfather',
   'year': '1972'}],
 [{'country': 'Estados Unidos',
   'director': 'Francis Ford Coppola',
   'duration': '200 min.',
   'rating': '8,9',
   'title': 'The Godfather Part IIaka',
   'year': '1974'}],
 [{'country': 'Estados Unidos',
   'director': 'David Simon',
   'duration': '60 min.',
   'rating': '8,8',
   'title': 'The Wireaka',
   'year': '2002'}],
 [{'country': 'Estados Unidos',
   'director': 'Vince Gilligan',
   'duration': '45 min.',
   'rating': '8,8',
   'title': 'Breaking Bad',
   'year': '2008'}],
 [{'country': 'Estados Unidos',
   'director': 'Carl Sagan',
   'duration': '60 min.',
   'rating': '8,8',
   'title': 'Cosmos',
   'year': '1980'}],
 [{'country': 'Esta

In [23]:
import pandas as pd

movies_flat = [movie[0] for movie in all_movies]

df = pd.DataFrame(movies_flat)

df

,title,year,country,director,duration,rating
0,The Godfather,1972,Estados Unidos,Francis Ford Coppola,175 min.,"9,0"
1,The Godfather,1972,Estados Unidos,Francis Ford Coppola,175 min.,"9,0"
2,The Godfather Part IIaka,1974,Estados Unidos,Francis Ford Coppola,200 min.,"8,9"
3,The Wireaka,2002,Estados Unidos,David Simon,60 min.,"8,8"
4,Breaking Bad,2008,Estados Unidos,Vince Gilligan,45 min.,"8,8"
5,Cosmos,1980,Estados Unidos,Carl Sagan,60 min.,"8,8"
6,Blue Planet II,2017,Estados Unidos,David Attenborough,60 min.,"8,7"
7,12 Angry Menaka,1957,Estados Unidos,Sidney Lumet,95 min.,"8,7"
8,Cosmos: A Space-Time Odysseyaka,2014,Estados Unidos,Ann Druyan,60 min.,"8,6"
9,Schindler's List,1993,Estados Unidos,Steven Spielberg,195 min.,"8,6"


Entramos en el link de las películas en cartelera

In [15]:
url2 = 'https://www.filmaffinity.com/es/cat_new_th_es.html'

driver.get(url2)

Sacamos todas las películas y llamamos a la función con cada película

Ahora usamos los links para llamar a la funcion y sacar los datos